In [0]:
storage_account_name = "stecommercepipeline"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

orders = spark.read.format("delta").load(silver_path + "orders")
customers = spark.read.format("delta").load(silver_path + "customers")
products = spark.read.format("delta").load(silver_path + "products")

In [0]:
from pyspark.sql.functions import col, count, when

def dq_report(df, table_name, key_col):
    total = df.count()
    nulls = df.filter(col(key_col).isNull()).count()
    dupes = total - df.dropDuplicates([key_col]).count()
    return (table_name, total, nulls, dupes)

results = []
results.append(dq_report(orders, "orders", "order_id"))
results.append(dq_report(customers, "customers", "customer_id"))
results.append(dq_report(products, "products", "product_id"))

dq_df = spark.createDataFrame(results, ["table_name", "total_rows", "null_keys", "duplicate_keys"])
dq_df.write.format("delta").mode("overwrite").save(gold_path + "data_quality_summary")
display(dq_df)

table_name,total_rows,null_keys,duplicate_keys
orders,99441,0,0
customers,99441,0,0
products,32951,0,0
